# SVM Individual Hyperparameter Trial - C

This notebook tests the SVM regularisation parameter `C` independently while keeping `kernel="rbf"`, `gamma="scale"`, and `class_weight="balanced"` fixed.

Labels:
- `0 = bona-fide`
- `1 = synthetic/deepfake`

This notebook is self-contained and preserves the same preprocessing, MFCC representation, split, scaling, and SVM training cap as the source notebook. The held-out test split is **not used** during this trial.

## 1. Environment Setup

Run this notebook from a clean Google Colab session or from the repository.


## 2. Package Installation


This cell checks whether the required Python packages are available and quietly installs any that are missing, allowing the rest of the notebook to run with the expected audio-processing, data-analysis, visualisation, and machine-learning libraries.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("joblib", "joblib"),
]

missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already installed.")


## 3. Imports


This cell imports the standard-library and third-party tools used throughout the notebook, including utilities for paths and reproducibility, audio feature extraction, data handling, plotting, preprocessing, model training, evaluation, and notebook-friendly display.

In [ ]:
import os
import random
from pathlib import Path

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from IPython.display import display
except Exception:
    display = print


## 4. Configuration and Random Seeds


This cell fixes the random seeds for reproducible results and defines the main experiment settings, including the audio duration and sample rate, MFCC parameters, supported file types, class labels, optional per-class sampling limit, and maximum SVM training size.

In [ ]:
RANDOM_STATE = 42
# Fixed random state used to make results reproducible across runs.

random.seed(RANDOM_STATE)
# Sets Python's built-in random number generator seed.

np.random.seed(RANDOM_STATE)
# Sets NumPy's random number generator seed.

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
# Sets Python's hash seed to improve reproducibility between runs.


SAMPLE_RATE = 22050
# Target audio sample rate in Hertz (22.05 kHz).

FIXED_DURATION_SECONDS = 5.0
# Fixed audio duration in seconds. Audio will be padded or truncated to 5 seconds.

N_MFCC = 40
# Number of Mel-Frequency Cepstral Coefficients (MFCCs) extracted from each audio sample.

N_FFT = 1024
# Number of audio samples used for each Fast Fourier Transform (FFT) frame.

HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
# Number of samples between consecutive analysis frames.
# Corresponds to a 10 ms hop at a 22.05 kHz sample rate.

WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))
# Length of each analysis window in samples.
# Corresponds to a 25 ms window at a 22.05 kHz sample rate.

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
# Supported audio file extensions that will be included during dataset scanning.

CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
# Maps numerical class labels to their human-readable class names.
# 0 = genuine/bona-fide speech, 1 = synthetic/deepfake speech.

MAX_FILES_PER_CLASS = None
# Maximum number of audio files to use from each class.
# None means that all available files are used.
# Set to a small value, such as 40, when performing a quick smoke test.

SVM_MAX_TRAIN_SAMPLES = 20000
# Maximum number of training samples used by the SVM model.
# This limits memory usage and training time when working with large datasets.

## 5. Dataset Paths


This cell mounts Google Drive when the notebook is running in Colab, selects local or Drive-based locations for the synthetic and bona-fide audio, creates the output folders for figures, models, and tables, and prints the paths that will be used.

In [ ]:
def mount_drive_if_colab():
    # Mount Google Drive only when the notebook is running inside Google Colab.
    try:
        from google.colab import drive
        # Import the Google Drive mounting utility provided by Colab.

        drive.mount("/content/drive")
        # Mount Google Drive so files stored in MyDrive can be accessed.

    except Exception:
        # Prevent the notebook from failing when it is executed outside Google Colab.
        print("Not running in Colab, or Drive is already available.")


mount_drive_if_colab()
# Attempt to mount Google Drive before defining dataset and output paths.


local_synthetic = Path.cwd().parent / "Datasets" / "Unprocessed" / "MLAAD_10pct"
# Define the expected local path of the MLAAD 10% synthetic audio subset.
# Path.cwd().parent refers to the parent directory of the current working directory.

bona_fide = Path.cwd().parent / "Datasets" / "Unprocessed" / "M_AILABS_bona_fide_subset"
# Define the expected local path of the MLAAD 10% synthetic audio subset.
# Path.cwd().parent refers to the parent directory of the current working directory.


SYNTHETIC_AUDIO_DIR = (
    local_synthetic
    if local_synthetic.exists()
    else Path(
        "/content/drive/MyDrive/Colab Notebooks/Education/INM701/"
        "Datasets/MLAAD_10pct"
    )
)
# Select the synthetic audio dataset location.
# If the dataset exists locally, use the local directory.
# Otherwise, use the MLAAD dataset stored in Google Drive.


BONA_FIDE_AUDIO_DIR = (
    bona_fide
    if bona_fide.exists()
    else Path(
        "/content/drive/MyDrive/Colab Notebooks/Education/INM701/"
        "Datasets/M_AILABS_bona_fide_subset"
    )
)
# Directory containing bona-fide (genuine/human) speech recordings.
# This must be changed to the appropriate bona-fide dataset path before
# performing the final binary classification training.


OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs"
)
# Define the main Google Drive directory where SVM experiment outputs
# such as figures, trained models, and result tables will be saved.


if not Path("/content").exists():
    OUTPUT_ROOT = Path.cwd() / "outputs" / "svm"
# If the notebook is not running in Google Colab, use a local output
# directory instead of the Google Drive path.


FIGURE_DIR = OUTPUT_ROOT / "figures"
# Directory used to save generated plots and evaluation figures.

MODEL_DIR = OUTPUT_ROOT / "models"
# Directory used to save trained models and related model files.

TABLE_DIR = OUTPUT_ROOT / "tables"
# Directory used to save evaluation results, metrics, and other tables.


for folder in [FIGURE_DIR, MODEL_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
# Create all required output directories.
# parents=True creates any missing parent directories.
# exist_ok=True prevents an error if a directory already exists.


print("Synthetic audio:", SYNTHETIC_AUDIO_DIR)
# Display the synthetic/deepfake audio directory being used.

print("Bona-fide audio:", BONA_FIDE_AUDIO_DIR)
# Display the genuine/human speech directory being used.

print("Output folder:", OUTPUT_ROOT)
# Display the directory where experiment outputs will be stored.

## 6. Dataset Loading / Audio File Scan


This cell defines a recursive audio-file scanner that builds metadata from the dataset folder structure, creates and optionally samples the synthetic and bona-fide manifests, verifies that both classes are present, saves the combined manifest, and displays a preview.

In [ ]:
def scan_audio_files(root, label):
    # Scan an audio directory and create one metadata row for each supported audio file.

    if root is None:
        # Return an empty DataFrame when no directory has been provided.
        # This allows the function to handle an unset dataset path safely.
        return pd.DataFrame()

    root = Path(root)
    # Convert the supplied directory path into a pathlib Path object.

    if not root.exists():
        # Stop execution if the specified audio directory cannot be found.
        raise FileNotFoundError(f"Audio folder not found: {root}")

    rows = []
    # Store the metadata extracted for each audio file.

    for path in sorted(root.rglob("*")):
        # Recursively search through the dataset directory and its subdirectories.

        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
            # Ignore directories and files that do not have a supported audio extension.
            continue

        relative_parts = path.relative_to(root).parts
        # Split the file's relative path into folder components.
        # These folder names are used to infer metadata such as language
        # and the TTS generator used to produce synthetic speech.


        if (
            label == 1
            and len(relative_parts) >= 3
            and relative_parts[0].lower() == "fake"
        ):
            # Handle MLAAD structures containing an explicit "fake" directory,
            # for example: fake/english/xtts/audio.wav.

            language = relative_parts[1]
            # Extract the language from the folder immediately after "fake".

            tts_generator = relative_parts[2]
            # Extract the TTS/deepfake generator from the following folder.


        elif label == 1 and len(relative_parts) >= 2:
            # Handle synthetic datasets where the structure starts directly
            # with the language and generator folders,
            # for example: english/xtts/audio.wav.

            language = relative_parts[0]
            # Extract the language from the first directory.

            tts_generator = relative_parts[1]
            # Extract the synthetic speech generator from the second directory.


        else:
            # Handle bona-fide (genuine/human) speech recordings.

            language = (
                relative_parts[0]
                if len(relative_parts) >= 2
                else "unknown"
            )
            # Infer the language from the first directory when possible.
            # Use "unknown" when the folder structure does not contain
            # sufficient information.

            tts_generator = "bona_fide"
            # Bona-fide speech has no TTS generator, so a fixed identifier is used.


        rows.append(
            {
                "path": str(path),
                # Store the complete path to the audio file.

                "relative_path": path.relative_to(root).as_posix(),
                # Store the path relative to the dataset root for portability.

                "label": label,
                # Store the numerical class label:
                # 0 = bona-fide speech, 1 = synthetic speech.

                "class_name": CLASS_NAMES[label],
                # Convert the numerical class label into its readable class name.

                "language": language,
                # Store the language inferred from the directory structure.

                "tts_generator": tts_generator,
                # Store the synthetic speech generator or "bona_fide"
                # for genuine speech.

                "source_file": path.stem,
                # Store the audio filename without its file extension.
            }
        )

    return pd.DataFrame(rows)
    # Convert all collected metadata rows into a Pandas DataFrame.


synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, label=1)
# Scan the MLAAD synthetic speech directory and assign class label 1.


bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, label=0)
# Scan the genuine human speech directory and assign class label 0.


manifest = pd.concat(
    [bona_fide_manifest, synthetic_manifest],
    ignore_index=True
)
# Combine the bona-fide and synthetic metadata into one dataset manifest.
# ignore_index=True creates a new continuous row index.


if MAX_FILES_PER_CLASS is not None:
    # Optionally restrict the number of files used from each class.
    # This is useful for quick tests before running the complete experiment.

    manifest = (
        manifest.groupby("label", group_keys=False)
        # Separate the manifest according to the binary class label.

        .apply(
            lambda group: group.sample(
                min(len(group), MAX_FILES_PER_CLASS),
                random_state=RANDOM_STATE
            )
        )
        # Randomly sample up to MAX_FILES_PER_CLASS examples from each class.
        # RANDOM_STATE ensures that the same samples are selected between runs.

        .sample(frac=1.0, random_state=RANDOM_STATE)
        # Shuffle the combined sampled dataset.

        .reset_index(drop=True)
        # Reset the DataFrame index after sampling and shuffling.
    )


if manifest.empty or set(manifest["label"].unique()) != {0, 1}:
    # Verify that the manifest is not empty and contains both required classes.

    raise RuntimeError(
        "Both classes are required. Set BONA_FIDE_AUDIO_DIR to the human speech folder "
        "and SYNTHETIC_AUDIO_DIR to the MLAAD synthetic folder."
    )
    # Stop execution when either the bona-fide or synthetic class is missing.
    # Binary SVM training requires examples from both classes.


manifest_path = TABLE_DIR / "svm_manifest.csv"
# Define the output path for the generated dataset manifest.


manifest.to_csv(manifest_path, index=False)
# Save the complete manifest as a CSV file for reproducibility
# and later inspection of the files used in the experiment.


display(manifest.head())
# Display the first five rows to verify that paths, labels,
# languages, and generator metadata were extracted correctly.

## 7. Dataset Inspection


This cell inspects the assembled dataset by reporting the total number of files and displaying counts by class, language, and speech generator so that the dataset composition can be checked before modelling.

In [ ]:
print("Total files:", len(manifest))
# Display the total number of audio files included in the dataset manifest.


display(
    manifest["class_name"]
    .value_counts()
    .rename("files")
)
# Count and display the number of files in each class.
# This helps verify the balance between bona-fide and synthetic speech samples.


display(
    manifest.groupby(["class_name", "language"])
    .size()
    .reset_index(name="files")
    .head(20)
)
# Group the dataset by class and language, then count the number of files
# in each combination.
# Display the first 20 rows to inspect the language distribution across classes.


display(
    manifest.groupby(["class_name", "tts_generator"])
    .size()
    .reset_index(name="files")
    .head(20)
)
# Group the dataset by class and TTS generator, then count the number of files
# associated with each generator.
# Bona-fide samples use the fixed "bona_fide" generator label.
# Display the first 20 rows to inspect the generator distribution.

## 8. Data Quality Checks


This cell creates a compact data-quality table that checks the manifest size, missing paths, files absent from disk, duplicate paths, and the number of samples in each class, then saves and displays the results.

In [ ]:
quality_checks = pd.DataFrame(
    [
        {"check": "rows", "value": len(manifest)},
        # Record the total number of audio files included in the manifest.

        {"check": "missing_paths", "value": int(manifest["path"].isna().sum())},
        # Count how many rows have a missing or undefined file path.

        {
            "check": "missing_files",
            "value": int(
                (~manifest["path"].map(lambda p: Path(p).exists())).sum()
            ),
        },
        # Check whether each file path actually exists on disk.
        # Count the number of manifest entries that point to missing files.

        {
            "check": "duplicate_paths",
            "value": int(manifest["path"].duplicated().sum()),
        },
        # Count duplicated file paths to ensure the same audio file
        # has not been included multiple times.

        {
            "check": "bona_fide_files",
            "value": int((manifest["label"] == 0).sum()),
        },
        # Count the number of bona-fide (genuine/human) speech samples.

        {
            "check": "synthetic_files",
            "value": int((manifest["label"] == 1).sum()),
        },
        # Count the number of synthetic/deepfake speech samples.
    ]
)
# Store the dataset quality-control results in a Pandas DataFrame.


quality_path = TABLE_DIR / "svm_data_quality_checks.csv"
# Define the output path for the dataset quality-check results.


quality_checks.to_csv(quality_path, index=False)
# Save the quality-check results as a CSV file for reproducibility
# and later inspection.


display(quality_checks)
# Display the quality-check table to verify that the manifest contains
# valid files, no unexpected duplicates, and both required classes.

## 9. Audio Preprocessing


This cell defines the audio preprocessing routine, which loads each recording as mono audio at the target sample rate, pads or truncates it to five seconds, applies peak normalisation, and checks the resulting shape and duration on one example.

In [ ]:
def load_audio_fixed(path):
    # Load an audio file as mono, resample it to the target sample rate,
    # and ensure that every sample has the same fixed duration.

    target_samples = int(round(SAMPLE_RATE * FIXED_DURATION_SECONDS))
    # Calculate the required number of audio samples based on the
    # target sample rate and fixed duration.

    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    # Load the audio file using Librosa.
    # sr=SAMPLE_RATE resamples the audio to 22.05 kHz.
    # mono=True converts multi-channel audio to a single mono channel.

    audio = np.asarray(audio, dtype=np.float32)
    # Convert the loaded audio waveform to a NumPy array using
    # 32-bit floating-point values.


    if len(audio) < target_samples:
        # Check whether the audio recording is shorter than the required duration.

        audio = np.pad(audio, (0, target_samples - len(audio)))
        # Pad shorter recordings with zeros (silence) at the end until
        # they reach the required number of samples.

    else:
        audio = audio[:target_samples]
        # Truncate recordings that are longer than the required duration
        # so that all audio samples have an identical length.


    peak = np.max(np.abs(audio))
    # Find the maximum absolute amplitude in the waveform.

    if peak > 0:
        audio = audio / peak
        # Apply peak normalization by scaling the waveform so that its
        # maximum absolute amplitude is 1.
        # The condition prevents division by zero for completely silent audio.


    return audio
    # Return the processed fixed-length and normalized audio waveform.


example_audio = load_audio_fixed(manifest.iloc[0]["path"])
# Load and preprocess the first audio file in the dataset manifest
# to verify that the preprocessing function operates correctly.


print("Example audio shape:", example_audio.shape)
# Display the number of samples in the processed audio waveform.


print("Example duration seconds:", len(example_audio) / SAMPLE_RATE)
# Calculate and display the duration of the processed waveform in seconds.
# This should match FIXED_DURATION_SECONDS (5 seconds).

## 10. MFCC Feature Extraction


This cell defines how MFCC matrices are extracted from the preprocessed audio and summarised into fixed-length vectors using each coefficient's mean and standard deviation, then verifies both feature shapes with an example file.

In [ ]:
def extract_mfcc_features(path):
    # Extract MFCC features from a preprocessed audio file.

    audio = load_audio_fixed(path)
    # Load the audio using the previously defined preprocessing function.
    # Each waveform is resampled, normalized, and fixed to the same duration.

    return librosa.feature.mfcc(
        y=audio,
        # Provide the preprocessed audio waveform.

        sr=SAMPLE_RATE,
        # Specify the audio sample rate used during MFCC extraction.

        n_mfcc=N_MFCC,
        # Extract 40 Mel-Frequency Cepstral Coefficients (MFCCs)
        # from each time frame.

        n_fft=N_FFT,
        # Set the FFT size used to compute the frequency spectrum
        # for each analysis frame.

        hop_length=HOP_LENGTH,
        # Set the distance between consecutive analysis frames.
        # This corresponds to approximately 10 ms.

        win_length=WIN_LENGTH,
        # Set the analysis window length.
        # This corresponds to approximately 25 ms.

        center=False,
        # Prevent Librosa from padding the audio around each frame.
        # Frames are calculated directly from the available waveform.

    ).astype(np.float32)
    # Convert the resulting MFCC matrix to 32-bit floating-point values.
    # The output shape is: (number of MFCC coefficients, number of time frames).


def aggregate_mfcc(mfcc):
    # Convert the two-dimensional MFCC matrix into one fixed-length
    # feature vector suitable for traditional machine-learning models such as SVM.

    return np.concatenate(
        [
            mfcc.mean(axis=1),
            # Calculate the mean value of each MFCC coefficient across time.
            # This provides information about the average spectral characteristics.

            mfcc.std(axis=1),
            # Calculate the standard deviation of each MFCC coefficient across time.
            # This captures how much each coefficient varies throughout the audio.
        ]
    ).astype(np.float32)
    # Concatenate the mean and standard deviation values into one vector.
    # With 40 MFCC coefficients, this produces 80 features per audio file.


example_mfcc = extract_mfcc_features(manifest.iloc[0]["path"])
# Extract the MFCC matrix from the first audio file in the manifest
# to verify that feature extraction is working correctly.


example_vector = aggregate_mfcc(example_mfcc)
# Aggregate the MFCC matrix into the fixed-length feature vector
# that will later be provided to the SVM classifier.


print("MFCC shape:", example_mfcc.shape)
# Display the dimensions of the MFCC matrix.
# Expected format: (40 MFCC coefficients, number of time frames).


print("Aggregated feature vector shape:", example_vector.shape)
# Display the dimensions of the final SVM feature vector.
# With N_MFCC = 40, the expected shape is (80,).

## 11. Label Preparation


This cell converts the manifest labels into an integer NumPy array and prints both the class-name mapping and the number of samples assigned to each class.

In [ ]:
y = manifest["label"].astype(int).to_numpy()
# Extract the class labels from the manifest.
# Convert them to integers and then to a NumPy array for use by
# the machine-learning models and dataset splitting functions.


print("Label mapping:", CLASS_NAMES)
# Display the mapping between numerical labels and class names.
# 0 = bona_fide speech and 1 = synthetic/deepfake speech.


print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))
# Count the number of samples belonging to each numerical class label.
# np.unique() returns the unique labels and their corresponding frequencies.
# zip() combines these into label-count pairs, which are converted into
# a dictionary for easier inspection.

## 12. Train / Validation / Test Split


This cell creates reproducible stratified training, validation, and test partitions in a 70:15:15 ratio, saves and displays their class distributions, and checks that no audio path appears in more than one split.

In [ ]:
train_manifest, temp_manifest = train_test_split(
    manifest,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=manifest["label"],
)
# Split the complete dataset into 70% training data and 30% temporary data.
# stratify=manifest["label"] preserves the class distribution of
# bona-fide and synthetic samples in both subsets.
# RANDOM_STATE ensures that the same split can be reproduced.


validation_manifest, test_manifest = train_test_split(
    temp_manifest,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_manifest["label"],
)
# Split the remaining 30% equally into validation and test sets.
# This produces an overall 70% training, 15% validation, and 15% test split.
# Stratification again preserves the class balance in both subsets.


train_manifest = train_manifest.reset_index(drop=True)
# Reset the training DataFrame index after splitting.

validation_manifest = validation_manifest.reset_index(drop=True)
# Reset the validation DataFrame index after splitting.

test_manifest = test_manifest.reset_index(drop=True)
# Reset the test DataFrame index after splitting.


split_summary = pd.concat(
    [
        train_manifest.assign(split="train"),
        # Add a column identifying all training samples.

        validation_manifest.assign(split="validation"),
        # Add a column identifying all validation samples.

        test_manifest.assign(split="test"),
        # Add a column identifying all test samples.
    ]
).groupby(["split", "class_name"]).size().unstack(fill_value=0)
# Combine all three subsets, group them by dataset split and class,
# and count the number of files belonging to each class.
# unstack() converts the class counts into separate table columns.
# fill_value=0 replaces any missing combinations with zero.


split_path = TABLE_DIR / "svm_split_summary.csv"
# Define the output path for the dataset split summary table.


split_summary.to_csv(split_path)
# Save the training, validation, and test class distributions as a CSV file
# for reproducibility and later inspection.


display(split_summary)
# Display the split summary to verify that the expected class
# distribution has been preserved across all three subsets.


for left_name, left_df, right_name, right_df in [
    ("train", train_manifest, "validation", validation_manifest),
    ("train", train_manifest, "test", test_manifest),
    ("validation", validation_manifest, "test", test_manifest),
]:
    # Compare every pair of dataset splits to check for duplicate file paths.

    overlap = set(left_df["path"]) & set(right_df["path"])
    # Compute the intersection between the file paths in the two subsets.
    # Any files appearing in both sets would indicate data leakage.

    print(f"Path overlap {left_name}-{right_name}: {len(overlap)}")
    # Display the number of overlapping audio files between each pair.
    # The expected result is 0 for all comparisons.

## 13. Aggregated MFCC Feature Preparation


This cell defines a routine that extracts aggregated MFCC features and labels for every file in a dataset split while recording failures, then builds and reports the training and validation feature matrices.

In [ ]:
def build_feature_table(split_manifest):
    # Extract one fixed-length aggregated MFCC feature vector
    # and corresponding class label for every file in a dataset split.

    features = []
    # Store the extracted feature vectors.

    labels = []
    # Store the corresponding numerical class labels.

    failed_paths = []
    # Store information about any files that fail during feature extraction.


    for _, row in split_manifest.iterrows():
        # Iterate through every audio file listed in the supplied manifest.

        try:
            mfcc = extract_mfcc_features(row["path"])
            # Load the audio file and extract its MFCC feature matrix.

            features.append(aggregate_mfcc(mfcc))
            # Convert the MFCC matrix into a fixed-length vector
            # using the mean and standard deviation of each coefficient.

            labels.append(int(row["label"]))
            # Store the corresponding binary class label:
            # 0 = bona-fide speech, 1 = synthetic speech.

        except Exception as exc:
            # Catch errors from unreadable, corrupted, or otherwise
            # problematic audio files without stopping the entire process.

            failed_paths.append(
                {
                    "path": row["path"],
                    "error": repr(exc),
                }
            )
            # Record both the failed file path and the associated error message
            # so that problematic files can be inspected later.


    if failed_paths:
        # Save a report only when one or more files failed.

        failed_df = pd.DataFrame(failed_paths)
        # Convert the recorded failures into a Pandas DataFrame.

        failed_df.to_csv(
            TABLE_DIR / "svm_feature_extraction_failures.csv",
            index=False
        )
        # Save the failure report as a CSV file for later inspection.

        print("Feature extraction failures:", len(failed_df))
        # Display the total number of files that could not be processed.


    return (
        np.vstack(features).astype(np.float32),
        np.asarray(labels, dtype=int),
    )
    # Stack all individual feature vectors into a two-dimensional
    # feature matrix with shape (number of files, number of features).
    # Also return the corresponding class labels as a NumPy integer array.


X_train, y_train = build_feature_table(train_manifest)
# Extract aggregated MFCC features and labels from the training split.


X_validation, y_validation = build_feature_table(validation_manifest)
# Extract aggregated MFCC features and labels from the validation split.


print("Train feature matrix:", X_train.shape)
# Display the dimensions of the training feature matrix.
# With 40 MFCC coefficients summarized by mean and standard deviation,
# each audio file should produce 80 features.


print("Validation feature matrix:", X_validation.shape)
# Display the dimensions of the validation feature matrix
# to confirm that feature extraction completed successfully.

## 14. Train-only Scaling


This cell fits a standardisation transform using only the training features and applies the same learned scaling to the validation features, preventing validation information from influencing preprocessing.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)

print("Scaler fitted on training data only.")


## 15. SVM Evaluation Helper and Fixed Training Subset

This cell defines the common evaluation function and constructs the same reproducible SVM training subset used across all individual hyperparameter trials.

In [ ]:
def evaluate_classifier(model, X, y_true, variant, split_name):
    # Evaluate a trained classifier and return the main binary-classification metrics.
    y_pred = model.predict(X)

    return {
        "variant": variant,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }, y_pred


# Use the same reproducible training subset in every SVM trial.
if len(y_train) > SVM_MAX_TRAIN_SAMPLES:
    selected = np.random.default_rng(RANDOM_STATE).choice(
        len(y_train),
        size=SVM_MAX_TRAIN_SAMPLES,
        replace=False,
    )
    X_fit = X_train_scaled[selected]
    y_fit = y_train[selected]
    print(f"Training SVM trial on {SVM_MAX_TRAIN_SAMPLES} sampled training rows.")
else:
    X_fit = X_train_scaled
    y_fit = y_train
    print(f"Training SVM trial on all {len(y_fit)} training rows.")


## 16. Individual Hyperparameter Trial — C

Only **C** is varied in this notebook. All other SVM settings are held fixed so the effect of this hyperparameter can be inspected independently on the validation split.

In [ ]:
# Only the target hyperparameter varies in this notebook.
svm_candidates = [
    {"variant": "C_0.01", "kernel": "rbf", "C": 0.01, "gamma": "scale", "class_weight": "balanced"},
    {"variant": "C_0.1",  "kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": "balanced"},
    {"variant": "C_1",    "kernel": "rbf", "C": 1.0,  "gamma": "scale", "class_weight": "balanced"},
    {"variant": "C_10",   "kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced"},
    {"variant": "C_100",  "kernel": "rbf", "C": 100.0,"gamma": "scale", "class_weight": "balanced"},
]

validation_rows = []
trained_models = {}

for params in svm_candidates:
    model = SVC(
        kernel=params["kernel"],
        C=params["C"],
        gamma=params["gamma"],
        class_weight=params["class_weight"],
        random_state=RANDOM_STATE,
    )
    model.fit(X_fit, y_fit)

    metrics, _ = evaluate_classifier(
        model,
        X_validation_scaled,
        y_validation,
        params["variant"],
        "validation",
    )
    metrics.update({k: v for k, v in params.items() if k != "variant"})
    validation_rows.append(metrics)
    trained_models[params["variant"]] = model

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

validation_path = TABLE_DIR / "svm_c_trial.csv"
validation_results.to_csv(validation_path, index=False)
display(validation_results)

best_variant = validation_results.iloc[0]["variant"]
best_model = trained_models[best_variant]
best_value = validation_results.iloc[0]["C"]

print("Best C:", best_value)
print("Selected validation variant:", best_variant)
print("Best validation F1:", validation_results.iloc[0]["f1"])

# Save the best validation-stage model from this individual trial.
trial_model_path = MODEL_DIR / "svm_c_trial_best_validation_model.joblib"
joblib.dump(best_model, trial_model_path)
print("Saved trial results:", validation_path)
print("Saved best validation model:", trial_model_path)


## 17. Validation F1 Plot

This plot provides a compact visual comparison of validation F1 across the values tested in this individual trial.

In [ ]:
plot_results = validation_results.copy()
plot_results["plot_value"] = plot_results["C"].astype(str)

plt.figure(figsize=(7, 4))
plt.plot(plot_results["plot_value"], plot_results["f1"], marker="o")
plt.xlabel("C")
plt.ylabel("Validation F1")
plt.title("SVM C Trial")
plt.xticks(rotation=30)
plt.tight_layout()

plot_path = FIGURE_DIR / "svm_c_trial.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved plot:", plot_path)


## 18. Reproducibility Checks

In [ ]:
print("Notebook:", "SVM Individual Hyperparameter Trial — C")
print("Self-contained: Yes")
print("Requires custom helper file: No")
print("Random state:", RANDOM_STATE)
print("Feature representation: MFCC mean and standard deviation per coefficient")
print("Split policy: 70% train, 15% validation, 15% test, stratified by label")
print("Target hyperparameter varied:", "C")
print("Test split used in this notebook: No")
print("The test split remains untouched until the final selected-model notebook.")
